# Using the OpenAlex API

##  🎓 About OpenAlex

[OpenAlex](https://openalex.org/) is a great resource for searching and accessing **scholarly publication data** comprehensively. Furthermore, OpenAlex holds data about **authors, institutions and sources** and more [entities](https://docs.openalex.org/api-entities/entities-overview) relating to those publications. For this, it brings together data from a series well-known [data sources](https://docs.openalex.org/additional-help/faq#where-does-your-data-come-from).

And, what's maybe the best about it: OpenAlex data is published under the Creative Commons Public Domain Dedication ([CC0](https://creativecommons.org/publicdomain/zero/1.0/)) - waiving copyright at all!

**💻 Snapshots and API**

OpenAlex offers data [snapshots](https://docs.openalex.org/download-snapshot) on a regularly basis, but can accessed far more nuanced via [**REST API**](https://docs.openalex.org/how-to-use-the-api/api-overview). This notebook will give you a small overview about its basic functionality and the data which can be accessed there.

There is one thing which is helpful to understand and use OpenAlex' API: It consists of different endpoints representing the different entity types, OpenAlex provides data on:
* `/works` - scholarly publications like journal articles, book chapters etc.
* `/authors` - the contributors of these publications
* `/topics` - the subjects, which are treated by works
* `/institutions` - the institutions, which host contributors of publications like universities
* `/sources` - the sources which hold the single works, like journals, series, conferences, and repositories
* `/publishers`- companies and organizations that distribute works
* `/funders`- organizations that fund research


This notebook will show these endpoints, but not all possible actions. Despite this, OpenAlex API is constantly growing: To get it all, consult the [documentation](https://developers.openalex.org/).

**☝️ Caveat**

Gathering and integrating data of scholarly publications is a tough thing to do. Data sources are manifold and even original data is often incomplete. So, also data of aggregators like OpenAlex will not be 100% complete, free from errors or duplicates. **Handle all derivative numbers with care** and draw conclusions with respect to this imperfect data base. If possible, use other data sources to crosscheck your findings.

**🔑 Get a (free) API key**

Since **13 February 2026** the OpenAlex API requires an **API key** for all requests. The former "polite pool" based on the `mailto` parameter has been retired - the parameter is simply ignored now. The key is free: create an account and get your key at [openalex.org/settings/api](https://openalex.org/settings/api). It comes with a free daily usage budget (resetting at midnight UTC). See [Authentication](https://help.openalex.org/api/authentication/) and [Pricing](https://help.openalex.org/access/pricing/) for details.

**🗂️ Data version "Walden"**

Since November 2025 the API serves the rebuilt data version "Walden". Coverage and data quality improved, some fields were removed or deprecated (e.g. *Concepts* in favour of *Topics*, `last_known_institution` in favour of `last_known_institutions`), and new work types like `preprint` or `review` were introduced. Additional "expansion" records (mostly datasets and repository records) are excluded from API results by default. Numbers shown in this notebook may therefore differ from earlier versions. Check the [deprecations](https://developers.openalex.org/guides/deprecations) page for details.

In [ ]:
# Check Python version
!python --version

In [ ]:
# Install necessary libraries if not already available
#!pip install requests pandas

In [ ]:
# Import the necessary libraries
import json, os, requests
import pandas as pd

In [ ]:
# IMPORTANT: Provide your OpenAlex API key (https://openalex.org/settings/api)
# Preferably set it as environment variable OPENALEX_API_KEY instead of writing it into the notebook
api_key = os.environ.get('OPENALEX_API_KEY', '...')

## 📚 Works endpoint

The core of OpenAlex data are **scholarly publications**. These are are represented as ***Work* entities**, which can be retrieved via the `/works` endpoint. Single *Work* entities can be directly accessed by their OpenAlex identifier (starting with a "W" like `W3093430496`) or by their Digital Object Identifier (DOI) respectively. You can access a random publication with `https://api.openalex.org/works/random`.

To learn about the attributes a *Work* entity can have, take a look into the [Work](https://docs.openalex.org/about-the-data/work) object documentation.

We'll start by accessing a single *Work* entity aka a publication record. The code cell below queries the OpenAlex API (via it's `/works` endpoint). The base URL is completed with an identifier and enriched with your API key.

There are several examples you might try by commenting out the identifiers and adapt the request accordingly.

In [ ]:
base_url = 'https://api.openalex.org/works/'
params = {'api_key': api_key}

# Using different work identifiers
openalex_id = 'W3093430496'                               # doctoral thesis
#openalex_id = 'W2489129678'                              # monograph
#openalex_id = 'W4200635144'                             # conference publication
#doi = 'doi:10.1007/978-3-319-20319-5_4'                  # book part
#doi = 'doi:10.1007/10.22617/tim210529'                   # report

doi =  'doi:https://doi.org/10.1007/s10479-016-2314-1'    # green OA article
#doi = 'doi:https://doi.org/10.1371/journal.pone.0121874' # gold OA article
#doi = 'doi:https://doi.org/10.1007/s10489-020-02029-z'   # bronze OA article
#doi = 'https://doi.org/10.1134/s2079086415010053'         # closed article

    
# Adapt identifier if necessary: doi, openalex_id
r = requests.get(base_url + doi, params=params)
data = r.json()
type(data)

The data answer of the OpenAlex API is [JSON](https://en.wikipedia.org/wiki/JSON) formatted. Therefore, it is decoded with the `json()` command and written into an Python object named `data`. Resembling the JSON structure, this is a Python [dictionary](https://docs.python.org/3/tutorial/datastructures.html#dictionaries).

This request and decoding method will be used throughout the whole notebook. If a request fails, check `r.status_code` and `r.text` first - e.g. a missing or invalid API key or an exhausted daily budget.

The `list()` function gives you an overview of the available data fields (which are the *Work*'s attributes) in the dictionary by listing all its *keys*. Calling a certain attribute *value* can easily done with its key, for instance `data['id']`.

In [ ]:
# Overview of available data fields
list(data)

In [ ]:
# Calling the OpenAlex work ID field by its key ("id")
data['id']

In [ ]:
# Publication title
data['title']

In [ ]:
# All available identifiers, given as an own Python dictionary
data['ids']

In [ ]:
# Publication type
data['type']

In [ ]:
# Bibliographic information
data['biblio']

In [ ]:
# Full text availability: these fields changed with the "Walden" data version,
# so we read them with .get() to avoid errors if a field is not delivered anymore
{key: data.get(key) for key in ['has_fulltext', 'has_content', 'fulltext_origin']}

In [ ]:
# Bibliographic information of primary location (= publication medium/infrastructure)
# "issn_l" should be used preferably of all ISSNs while using OpenAlex API with ISSN identifier
data['primary_location']

In [ ]:
# Alternative locations, especially to provide Open Access versions of closed publications
data['locations']

In [ ]:
# Open Access status
data['open_access']

In [ ]:
# The first 5 cited works by their OpenAlex links
data['referenced_works'][:5]

In [ ]:
# Times cited
data['cited_by_count']

In [ ]:
# Citations by year (since 2012)
data['counts_by_year']

In [ ]:
# API link to citing publications
data['cited_by_api_url']

In [ ]:
# Status information: paratext, retraction
print(data['is_paratext'])
data['is_retracted']

A special feature of OpenAlex *Work* data is the attribute `abstract_inverted_index`. Instead of delivering the plain text **abstract of a publication**, its vocabulary or tokenized version is given, listing all words/tokens with their position in the abstract. This is done for copyright reasons. However, this information can be used to search *Works* (see 2.2) and build other applications upon.
Note that punctuation is treated as part of the tokens.

In [ ]:
# Inverted index with positions of words
data['abstract_inverted_index']

In [ ]:
# First 5 related works by their OpenAlex links
data['related_works'][:5]

In [ ]:
# How up to date is this information?
data['updated_date']

In [ ]:
# Primary topic (subject) with its hierarchy of subfield, field and domain
data['primary_topic']

In [ ]:
# Use Python list comprehension to access the display names and scores of all attached topics
[[i['display_name'], i['score']] for i in data['topics']]

In [ ]:
# How many MESH descriptors are there?
len(data['mesh'])

In [ ]:
# Show the first 5 MESH descriptors with qualifier ("None" = no qualifier)
[[i['descriptor_name'], i['qualifier_name']] for i in data['mesh'][:5]]

In [ ]:
# Showing the APC of the source journal
data['apc_list']

### Author and affiliation data

The *Works* object can hold attributes which provide more detailed information, or in a certain way, sub-attributes, like the attribute `authorships`.
Let's take a look at some `authorships` data: It holds all authors of the publication, each introduced by the `author_position` statement. Thereafter follows the individual author's data, resembling the *Author* entity will get to now later with its own endpoint.

Note that authors can have multiple `institutions` (which are their affiliations), each in turn described in detail, too.

In [ ]:
# Showing the first 2 authors, assembled in "authorships" attribute
data['authorships'][:2]

In [ ]:
# Print out the names, ORCIDs, raw affiliation strings and positions of all authors with a loop
for i in data['authorships']:
    print(i['author']['display_name'])
    print(i['author']['orcid'])
    print(i['raw_affiliation_string'])
    print(i['author_position'])
    print('---')

In [ ]:
# Print out all affiliations (short, normalized form) of the authors with a nested loop
for i in data['authorships']:
    for j in i['institutions']:
        print(j['display_name'])

### Use the search parameter

With a slight adjustment you can use the *Works* endpoint for a simple **search in the title, abstract fields** and **fulltext**, if available for OpenALex. You can search multiple terms just by separating them with a blank. You can also search for **exact phrases** using quotation marks.

The ranking (`relevance_score`) depends on similarity between request and field data, proximity of the single terms and citation counts. See [here](https://docs.openalex.org/api-entities/works/search-works) for a more detailed description of the search functionality.

In [ ]:
# Search 2 terms in title, abstract and fulltext, where available
url = 'https://api.openalex.org/works?search=einstein philosoph'

# Compare
#url = 'https://api.openalex.org/works?search=einsteins philosophie'

# Search an exact phrase using quotes ""
#url = 'https://api.openalex.org/works?search="einstein als philosoph"'

params = {'api_key': api_key}

r = requests.get(url, params=params)
data = r.json()

In [ ]:
data['meta']

In [ ]:
for i in data['results'][:10]:
    print(i['title'])
    print(i['relevance_score'])
    print(i['type'])
    print('---')

### Access multiple Work entities

After retrieving one specific publication record successfully, it's time to look for a bunch of publications. First of all, it is easy to **access all available works** with the OpenAlex API - just use the *Works* endpoint without any further parameters. But presumably you will need this amount of data only in exceptional cases.

But you can easily **add** to the *Work* endpoint **several filters** to narrow down your search. You find the **available filters** for *Work* entities [here](https://docs.openalex.org/api-entities/works/filter-works). As you may notice these are mainly the attributes and sub-attributes of the *Work* entities seen above.

#### Filter works by publication type

Let's start with filtering all available publications due to a **certain publication type**, like for instance book parts or reports. OpenAlex uses its own [list of work types](https://api.openalex.org/work-types) (e.g. `article`, `book-chapter`, `dissertation`, `preprint`, `review`); the original Crossref type is still available in the attribute `type_crossref`. Feel free to try them out in the  code cell below.

In [ ]:
# Accessing all publication of a certain publication type
#url = 'https://api.openalex.org/works'   # all available publications

url = 'https://api.openalex.org/works?filter=type:dissertation'
params = {'api_key': api_key}

r = requests.get(url, params=params) 
data = r.json()

In [ ]:
print(list(data))
data['meta']

In the `results` section you will find the matching records.

The `count` attribute in the `meta`section says how many results are there at all. The `per_page` attribute tells you how many records are delivered at once as default.

In [ ]:
# Pick one example from the results list
data['results'][11]

#### Filter works by "source":  journal, repository etc

Next we try to retrieve all publications of a certain "source". ***Source* entities** in OpenAlex are the venues, like **journals, conferences, preprint repositories,** and **institutional repositories**, that host publications. For journals articles this will be their journals, for book parts their volume. Sources can also be monograph series like [S4210167446](https://api.openalex.org/sources/S4210167446), repositories like [S4306402567](https://api.openalex.org/sources/S4306402567).

Source information is given in the *Works* attribute `primary_location` and `locations`. To get an impression of sources you may use the random generation `https://api.openalex.org/sources/random`.

To identify the desired source you can use its OpenAlex ID (starting with a "S"), for instance `S4210209919`. This identifier is also a sub-attribute of `locations.source`, and can be addressed with `locations.source.id` in the *Works* endpoint filter. You may also use the popular [ISSN](https://en.wikipedia.org/wiki/International_Standard_Serial_Number) identifier in the same way.

In [ ]:
# Accessing works of a certain host venue, using OpenAlex ID
url = 'https://api.openalex.org/works?filter=locations.source.id:S4210209919'

# Using ISSN identifier
#url = 'https://api.openalex.org/works?filter=locations.source.issn:2391-7652'

params = {'api_key': api_key}

r = requests.get(url, params=params) 
data = r.json()
data['meta']

In [ ]:
# Article titles of the result set
[i['title'] for i in data['results']]

We might adapt now the request by defining a **bigger default result set** to get all hits at once.

Instead to add the parameter to the URL directly, you may also use the `params` dictionary for more clarity.

In [ ]:
url = 'https://api.openalex.org/works?'

params = {'api_key': api_key,
          'filter': 'locations.source.id:S4210209919',
          'per_page': 100}

r = requests.get(url, params=params) 
data = r.json()

In [ ]:
data['meta']

In [ ]:
# Pick one example of the result set
data['results'][67]

In [ ]:
# Access 5 publications by title
[i['title'] for i in data['results'][35:40]]

In [ ]:
# Access the topics of a sample from the middle of the publication set
[[j['display_name'] for j in i['topics']] for i in data['results'][35:37]]

#### Filter works by author & handle big result sets

Similarly to the venue filtering, you may show all publication by a certain author, using its OpenAlex ID (starting with a "A"), for instance `A2420227930`. This identifier can be addressed with `authorships.author.id` or shorter, `author.id` in the *Works* endpoint filter.

Furthermore, you may access a certain sub result set ("result page") via the `page` parameter, like in the example below.

In [ ]:
url = 'https://api.openalex.org/works?filter=author.id:A5067955208&page=1&per-page=50'
params = {'api_key': api_key}

r = requests.get(url, params=params) 
data = r.json()
data['meta']

Since the `per-page` parameter only can be set up to 200, several requests are necessary to access the whole result set.

This can be easily done by writing a small loop.

In [ ]:
results = []
for page in range(1,5):
    url = 'https://api.openalex.org/works?'
    params = {'api_key': api_key,
              'filter': 'author.id:A5067955208',
              'per-page': 100,
              'page': page}
    r = requests.get(url, params=params)
    data = r.json()
    for record in data['results']:
        results.append(record)
len(results)

In [ ]:
[i['title'] for i in results[320:330]]

#### Group works due to a certain feature

Grouping is another way to adjust your data search. Here, you **split the result set** into groups **according to a certain data attribute**. Note that doing so, you are not targeting the records themselves, but deriving statements from the data. Because of this you will not use `filter` as parameter but `group_by` in your request. Accordingly, you will find the `results` section empty, but your demanded data in the `group_by`section.

For example you might look for the hosting sources of all publications, grouping them around this feature, and get the size of these groups. Note that the maximum of groups is 200 - if there are more variants you will get the 200 biggest of them.

In [ ]:
url = 'https://api.openalex.org/works'
params = {'api_key': api_key,
         'group_by': 'locations.source.id'}

r = requests.get(url, params=params) 
data = r.json()

In [ ]:
list(data)

In [ ]:
data['meta']

In [ ]:
data['group_by'][:5]

In [ ]:
# Accessing the biggest 20 sources, default sorted in descending count
[[i['key_display_name'], i['count']] for i in data['group_by'][:20]]

#### Combine filters and grouping

You can easily combine filters and grouping, for instance, filter according to a specific institution, like *University of Bern* `I118564535`, and group its publications by their main subject `primary_topic.id`. Then, of course, the resulting groups are based on the filtered subset.

In [ ]:
# Use the "params" dictionary for using filter and group_by at the same time
url = 'https://api.openalex.org/works'
params = {'api_key': api_key,
          'filter': 'institutions.id:I118564535',
          'group_by': 'primary_topic.id'}

r = requests.get(url, params=params)
data = r.json()

In [ ]:
# Accessing the 20 most frequent topics, default sorted in descending count
[[i['key_display_name'], i['count']] for i in data['group_by'][:20]]

In [ ]:
topics_data =[[i['key_display_name'], i['count']] for i in data['group_by']]
topics_sum = 0
for i in topics_data:
    topics_sum += i[1]
topics_sum

In [ ]:
# Calculating and adding percentages
for i in topics_data:
    i.append(str(round(i[1]/topics_sum*100,1)) + '%')
topics_data[:10]

## 👨‍🎓  Authors endpoint

Similar to the *Works* endpoint, there is an ***Authors*** endpoint in OpenAlex, delivering rich information about authors of scholarly publications: `/authors`. The *Authors* endpoint can be accessed via OpenAlex identifier (starting with an "A" like `A2156811675`) or for instance, via the popular [ORCID](https://orcid.org/) identifier. For detailed information see the *Author* object documentation [here](https://developers.openalex.org/api-reference/authors).


In [ ]:
base_url = 'https://api.openalex.org/authors/'
params = {'api_key': api_key} 

# Usable identifiers:
openalex_id = 'A2156811675'
openalex_id_namespace = 'openalex:A2156811675'
orcid = 'orcid:0000-0002-9344-1029'

# Adjust the used identifier
r = requests.get(base_url + orcid, params=params) 
data = r.json()

In [ ]:
list(data)

In [ ]:
data['id']

In [ ]:
# All available person identifiers
data['ids']

In [ ]:
# Normalized plain name and variants
print(data['display_name'])
data['display_name_alternatives']

In [ ]:
# How many times the publications of this person were cited?
print(data['cited_by_count'])
data['summary_stats']

In [ ]:
# Count of published works and citations per year, sicne 2012
data['counts_by_year']

In [ ]:
# API link to the publications of the author (using filter of the "Works" endpoint)
data['works_api_url']

In [ ]:
# The person's last known affiliations - a list, since authors can have several
[i['display_name'] for i in data['last_known_institutions']]

In [ ]:
# Topics the author is publishing about (replacing the deprecated "x_concepts")
[[i['display_name'], i['count']] for i in data['topics'][:10]]

### Use the search parameter

Using the search parameter on the `/authors`endpoint looks for the terms in the `display_name` attribute only. Read more about the search functionality [here](https://docs.openalex.org/api-entities/authors/search-authors).

In [ ]:
base_url = 'https://api.openalex.org/authors?'
params = {'api_key': api_key,
         'search': 'erik eriksen'}

r = requests.get(base_url, params=params)
data = r.json()

In [ ]:
data['meta']

In [ ]:
# Name, number of works and last known affiliations of "erik eriksen" authors
[[i['display_name'], i['works_count'], [j['display_name'] for j in i['last_known_institutions']]] \
                     for i in data['results'][:10]]

### Access multiple Author entities

Like for publications (aka *Works* entities), you can search for mutliple authors at once, and **use filters and groupings**. 

#### Filter authors by institution & put the data into a dataframe

For example, you might search for all authors, whose last known affiliations include *University of Bern*. For this, you can use its OpenAlex identifier `I118564535`. Filtering authors will use the `works_count` attribute to sort the results by default. To adjust the default sorting methods look [here](https://docs.openalex.org/how-to-use-the-api/get-lists-of-entities/sort-entity-lists).

In [ ]:
url = 'https://api.openalex.org/authors?'
params = {'api_key': api_key,
         'filter': 'last_known_institutions.id:I118564535',
         'per-page': 20}

r = requests.get(url, params=params) 
data = r.json()

In [ ]:
data['meta']

By reading out certain attributes of the result set you can easily transform the data into a dataframe for further research.

In [ ]:
names = []
ids = []
orcids = []
workcounts = []
citedcounts = []
workslinks = []

for i in data['results']:
    names.append(i['display_name'])
    ids.append(i['id'])
    orcids.append(i['orcid'])
    workcounts.append(i['works_count'])
    citedcounts.append(i['cited_by_count'])
    workslinks.append(i['works_api_url'])
    
zipped = list(zip(names, ids, orcids, workcounts, citedcounts, workslinks))
df = pd.DataFrame(zipped, columns=['author', 'openalex_id', 'orcid', \
                                    'works_count', 'cited_by_count', 'works_link'])

In [ ]:
df

#### Filter by multiple features & use logical operators

You can not only use several filters at once, by **simply stacking** them. An you can use filters with **logical operators**, like `!` for NOT, `|` for OR, less-than `<`, or greater-than `>`. For further details see [this page](https://docs.openalex.org/how-to-use-the-api/get-lists-of-entities/filter-entity-lists).

In [ ]:
# Example: Authors with only a few publications, but high citation count
url = 'https://api.openalex.org/authors?'
params = {'api_key': api_key,
         'filter': 'works_count:<5,cited_by_count:>50000',
         'per-page': 20}

r = requests.get(url, params=params) 
data = r.json()
data['meta']

In [ ]:
[[i['display_name'], i['works_count'], i['cited_by_count'], i['works_api_url']] for i in data['results']]

In [ ]:
# Example: Authors from companies or nonprofit institutions outside the USA and China
url = 'https://api.openalex.org/authors?'
params = {'api_key': api_key,
         'filter': 'last_known_institutions.type:company|nonprofit,last_known_institutions.country_code:!US|CN',
         'per-page': 20}

r = requests.get(url, params=params) 
data = r.json()
data['meta']

In [ ]:
[[i['display_name'], [j['display_name'] for j in i['last_known_institutions']]] for i in data['results']]

#### Group authors due to a certain feature

As a use case, it is very easy to group all authors according to their last known institution's country. Just use the `last_known_institutions.country_code` sub-attribute of the *Author* object as **grouping parameter**. Find more features to group authors by [here](https://docs.openalex.org/api-entities/authors/group-authors).

In [ ]:
url = 'https://api.openalex.org/authors?'
params = {'api_key': api_key,
         'group_by': 'last_known_institutions.country_code'}

r = requests.get(url, params=params) 
data = r.json()

In [ ]:
data['meta']

In [ ]:
# Accessing the 10 biggest countries of affiliations, default sorted in descending count
[[i['key_display_name'], i['count']] for i in data['group_by'][:10]]

#### Combine filters and grouping

Of course you can combine filters and groupings also for authors' data. In the example beneath you will group all authors writing about *Ethical Implications of Artificial Intelligence* `T10883` according to their (last known) institution type.

In [ ]:
url = 'https://api.openalex.org/authors?'
params = {'api_key': api_key,
         'filter': 'topics.id:T10883',
         'group_by': 'last_known_institutions.type'}

r = requests.get(url, params=params) 
data = r.json()
data['meta']

In [ ]:
data['group_by']

In [ ]:
inst_data =[[i['key_display_name'], i['count']] for i in data['group_by']]
inst_sum = 0
for i in inst_data:
    inst_sum += i[1]
inst_sum

In [ ]:
# Calculating and adding percentages
for i in inst_data:
    i.append(str(round(i[1]/inst_sum*100,1)) + '%')
inst_data

Now let's have a look at those authors writing about AI Ethics, whose last know affiliation is a company. For this, we will use 2 filters concurrently here: `topics.id:T10883` and `last_known_institutions.type:company`.

In [ ]:
url = 'https://api.openalex.org/authors?'
params = {'api_key': api_key,
         'filter': 'topics.id:T10883,last_known_institutions.type:company'}

r = requests.get(url, params=params) 
data = r.json()
data['meta']

In [ ]:
# Showing the 20 most productive authors in AI Ethics working at a company
[[i['display_name'], [j['display_name'] for j in i['last_known_institutions']], i['works_count']] \
                     for i in data['results'][:20]]

## 🔖 Topics endpoint

***Topics*** entities are OpenAlex' notion of subjects, which are attached to *Works*, using an automated system that takes into account the available information including title, abstract, source (journal) name, and citations. The highest-scoring topic is that Work's `primary_topic`. There is also an attribute `topics`of a Work entity with additional highly ranked topics.

Topics are hierarchically organzied in sub-fields, fields, and domains, for further information see the [Topics page](https://docs.openalex.org/api-entities/topics)


In [ ]:
base_url = 'https://api.openalex.org/topics/'
params = {'api_key': api_key} 

openalex_id = 'T10101'

r = requests.get(base_url + openalex_id, params=params) 
data = r.json()

In [ ]:
# Available attributes of the Topic entity
list(data)

In [ ]:
print(data['display_name'])
print(data['ids'])
data['keywords']

In [ ]:
data['description']

## 🏫 Institutions endpoint

***Institution*** entities are the organizations which host the authors when they write their publications (affiliations). Thus, publications can be assigned as output to certain institutions. OpenAlex uses the categories of institutions of the [ROR](https://ror.org/) registry: education, funder, healthcare, company, archive, nonprofit, government, facility, other. See also the [Institution](https://docs.openalex.org/about-the-data/institution) object documentation.

You can use the `/institutions` endpoint with OpenAlex identifier (starting with an "I" like `I118564535`) or the popular identifier ROR ID.

In [ ]:
base_url = 'https://api.openalex.org/institutions/'
params = {'api_key': api_key} 

#openalex_id = 'I118564535'
#openalex_id_namespace = 'openalex:I118564535'
ror_id = 'ror:02k7v4d05'

# Adapt identifier if necessary
r = requests.get(base_url + ror_id, params=params) 
data = r.json()

In [ ]:
# Overview of data fields
list(data)

In [ ]:
# OpenAlex institution ID
data['id']

In [ ]:
# All institution identifiers
data['ids']

In [ ]:
# Preferable name
data['display_name']

In [ ]:
# Display name in other languages, e.g. Latin
data['international']['display_name']['la']

In [ ]:
# API link to all affiliated publications
data['works_api_url']

In [ ]:
# Number of citations of affiliated publications
data['cited_by_count']

In [ ]:
# Number of affiliated publications and citations by year, since 2012
data['counts_by_year']

In [ ]:
# Associated institutions
data['associated_institutions']

In [ ]:
# Geographical information
data['geo']

### Access multiple Institution entities

You can access all *Institutions* at once, and use filters and groupings on them. See the available filters for *Institution* entities [here](https://docs.openalex.org/api-entities/institutions/filter-institutions) and possible grouping parameters [here](https://docs.openalex.org/api-entities/institutions/group-institutions).

In the example you will narrow down to Swiss institutions only and group these due to institution types.

In [ ]:
url = 'https://api.openalex.org/institutions?'
params = {'api_key': api_key,
         'filter': 'country_code:CH',
         'group_by': 'type'}

r = requests.get(url, params=params) 
data = r.json()

In [ ]:
data['meta']

In [ ]:
[[i['key_display_name'], i['count']] for i in data['group_by']]

Now, let's look at the Swiss archives producing scholarly publications. To get this data, you have to use 2 filters at once: `country_code:CH` and `type:archive`.

In [ ]:
url = 'https://api.openalex.org/institutions?'
params = {'api_key': api_key,
         'filter': 'country_code:CH,type:archive'}

r = requests.get(url, params=params) 
data = r.json()
data['meta']

In [ ]:
# Showing the top 10
[[i['display_name'], i['works_count']] for i in data['results'][:20]]

## 📰 Sources endpoint

***Sources*** entities in OpenAlex are the **journals, conferences, preprint repositories**, and **institutional repositories** that hold the single publications aka `Works`. For journal articles this will be their journals, for book parts their volume. Sources for instance can also be monograph series like `S4210167446` or repositories like `S2734324842`.

To access data about a certain *Source*, you can use its OpenAlex identifier (starting with a "S" like [S4210209919](https://openalex.org/S4210209919)) or the the popular [ISSN](https://en.wikipedia.org/wiki/International_Standard_Serial_Number) identifier. See also the [Source](https://docs.openalex.org/api-entities/sources) entity documentation and try a random source with `https://api.openalex.org/sources/random`.

In [ ]:
base_url = 'https://api.openalex.org/sources/'
params = {'api_key': api_key} 

#openalex_id = 'S98348390'
#issn = 'issn:0028-0836'   # non-OA journal
issn = 'issn:1654-6369'   # OA journal

r = requests.get(base_url + issn, params=params) 
data = r.json()

In [ ]:
list(data)

In [ ]:
# All identifiers. "issn_l" should be used preferably while using OpenAlex API.
data['ids']

In [ ]:
# Title of the source & name of host organization/publisher & website
print(data['display_name'])
print(data['host_organization_name'])
data['homepage_url']

In [ ]:
# Is it an Open Access venue (and is it in DOAJ registry)?
print(data['is_oa'])
data['is_in_doaj']

In [ ]:
# Number of hosted publications
data['works_count']

In [ ]:
# Topics
[[i['display_name'] for i in data['topics']]]

In [ ]:
# APC information
data['apc_prices']

In [ ]:
# API link to hosted publications
data['works_api_url']

### Access multiple Source entities

You can access all *Sources* at once, and use filters and groupings on them. See the available filters for *Source* entities [here](https://docs.openalex.org/api-entities/sources/filter-sources) and possible grouping parameters [here](https://docs.openalex.org/api-entities/sources/group-sources).

In the example at the end you will narrow down to open access sources only and group these due to publishers resp. `host_organization`.

In [ ]:
# Acessing all sources (sorted by number of works)
url = 'https://api.openalex.org/sources?'
params = {'api_key': api_key}

r = requests.get(url, params=params) 
data = r.json()
data['meta']

In [ ]:
# Showing the first 20 biggest sources (name, number of works, OA status)
[[i['display_name'], i['works_count'], i['is_oa']] for i in data['results'][:20]]

In [ ]:
# Inspecting OA sources according to their publishers
url = 'https://api.openalex.org/sources?'
params = {'api_key': api_key,
         'filter': 'is_oa:True',
         'group_by': 'host_organization'}

r = requests.get(url, params=params) 
data = r.json()
data['meta']

In [ ]:
pub_data = [[i['key_display_name'], i['count']] for i in data['group_by']]
pub_sum = 0
for i in pub_data:
    pub_sum += i[1]
pub_sum

In [ ]:
# Showing the 20 biggest OA publishers (according to number of sources), with percentages
for i in pub_data:
    i.append(str(round(i[1]/pub_sum*100,1)) + '%')
pub_data[:20]